# Tuần 05: Visualization cho paper

Mục tiêu hôm nay là tạo **hai figure paper-ready** từ cleaned data: một bar chart để so sánh mean gain, một dot plot để thấy individual learner records, rồi viết caption và interpretation không overclaim.

## 1. Setup

Cell này chỉ cần chạy. Nó tìm đúng thư mục Week 05, tải CSV nếu cần, và tạo thư mục output cho figures/tables.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

THIS_WEEK = "week-05-visualization-for-paper"
EXPECTED_SHA256 = "175469cd9120b36a467d0e0b439f78859525841555c6a30bc5de0777edb9137a"


def find_week_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "weeks" / THIS_WEEK,
        Path.cwd().parent,
        Path.cwd().parent / "weeks" / THIS_WEEK,
    ]
    for candidate in candidates:
        if candidate.name == THIS_WEEK and (candidate / "data/raw").exists():
            return candidate
        if (candidate / "data/raw/week05_cleaned_tcsol_scores.csv").exists():
            return candidate
    week_dir = Path.cwd() / "weeks" / THIS_WEEK
    week_dir.mkdir(parents=True, exist_ok=True)
    return week_dir


def course_path(path):
    path = Path(path)
    if THIS_WEEK in path.parts:
        start = path.parts.index(THIS_WEEK)
        return Path("weeks") / Path(*path.parts[start:])
    return path.name


WEEK_DIR = find_week_dir()
DATA_PATH = WEEK_DIR / "data/raw/week05_cleaned_tcsol_scores.csv"
TABLE_DIR = WEEK_DIR / "outputs/tables"
FIGURE_DIR = WEEK_DIR / "outputs/figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    source_url = "https://raw.githubusercontent.com/mtuann/tcsol-python-research/main/weeks/week-05-visualization-for-paper/data/raw/week05_cleaned_tcsol_scores.csv"
    urlretrieve(source_url, DATA_PATH)

actual_hash = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
if actual_hash != EXPECTED_SHA256:
    raise ValueError("Downloaded CSV does not match the expected Week 05 teaching dataset.")

sns.set_theme(style="whitegrid", font_scale=1.05)
print("pandas version:", pd.__version__)
print("matplotlib version:", matplotlib.__version__)
print("seaborn version:", sns.__version__)
print("Data file:", course_path(DATA_PATH))


pandas version: 2.3.3
matplotlib version: 3.9.4
seaborn version: 0.13.2
Data file: weeks/week-05-visualization-for-paper/data/raw/week05_cleaned_tcsol_scores.csv


## 2. Load data

Dataset này là cleaned export từ Week 04. Ta kiểm row count và các cột cần cho figure.

In [2]:
df = pd.read_csv(DATA_PATH)

print("Raw rows loaded:", len(df))
print("Columns:", list(df.columns))
print(df[["learner_id", "activity_focus", "pre_score", "post_score", "gain_score", "usable_pre_post"]].head(8).to_string(index=False))


Raw rows loaded: 36
Columns: ['learner_id', 'class_group', 'activity_focus', 'pre_score', 'post_score', 'attendance_hours', 'completed', 'self_confidence', 'class_group_raw', 'activity_focus_raw', 'completed_raw', 'gain_score', 'usable_pre_post']
learner_id     activity_focus  pre_score  post_score  gain_score  usable_pre_post
      S001      measure_words       62.0        75.0        13.0             True
      S002      measure_words       58.0        70.0        12.0             True
      S003      measure_words        NaN        72.0         NaN            False
      S004      measure_words       60.0         NaN         NaN            False
      S005 result_complements       55.0        68.0        13.0             True
      S006 result_complements       59.0        73.0        14.0             True
      S007 result_complements       61.0        74.0        13.0             True
      S008 result_complements        NaN        71.0         NaN            False


## 3. Filter usable records

Figure core chỉ dùng learner records có completed status và numeric pre/post scores. Đây là nơi caption lấy N.

In [3]:
usable = df[df["usable_pre_post"] == True].copy()

print("Usable rows:", len(usable))
print("Activity groups:", usable["activity_focus"].nunique())
print(usable["activity_focus"].value_counts().to_string())


Usable rows: 25
Activity groups: 4
activity_focus
vocabulary_review     8
result_complements    6
word_order            6
measure_words         5


## 4. Make labels readable

Ta đổi display labels bằng code. Dữ liệu gốc không đổi.

In [4]:
activity_labels = {
    "result_complements": "Result complements",
    "measure_words": "Measure words",
    "vocabulary_review": "Vocabulary review",
    "word_order": "Word order",
}
plot_labels = {
    "result_complements": "Result\ncomplements",
    "measure_words": "Measure\nwords",
    "vocabulary_review": "Vocabulary\nreview",
    "word_order": "Word\norder",
}
order = ["result_complements", "measure_words", "vocabulary_review", "word_order"]
usable["activity_label"] = usable["activity_focus"].map(activity_labels)
usable["plot_label"] = usable["activity_focus"].map(plot_labels)
usable["activity_label"] = pd.Categorical(usable["activity_label"], [activity_labels[x] for x in order], ordered=True)
usable["plot_label"] = pd.Categorical(usable["plot_label"], [plot_labels[x] for x in order], ordered=True)

print(usable[["activity_focus", "activity_label", "plot_label"]].drop_duplicates().to_string(index=False))


    activity_focus     activity_label          plot_label
     measure_words      Measure words      Measure\nwords
result_complements Result complements Result\ncomplements
        word_order         Word order         Word\norder
 vocabulary_review  Vocabulary review  Vocabulary\nreview


## 5. Summary table

Summary table là cách kiểm figure. Nếu figure nói mean gain, bảng phải có `n` và `mean_gain`.

In [5]:
summary = (
    usable.groupby("activity_label", observed=True)
    .agg(
        n=("learner_id", "count"),
        mean_gain=("gain_score", "mean"),
        median_gain=("gain_score", "median"),
        min_gain=("gain_score", "min"),
        max_gain=("gain_score", "max"),
    )
    .reset_index()
)
summary[["mean_gain", "median_gain", "min_gain", "max_gain"]] = summary[["mean_gain", "median_gain", "min_gain", "max_gain"]].round(2)
summary_path = TABLE_DIR / "week05_figure_summary.csv"
summary.to_csv(summary_path, index=False)

print(summary.to_string(index=False))
print("Saved summary table to:", course_path(summary_path))


    activity_label  n  mean_gain  median_gain  min_gain  max_gain
Result complements  6      13.00         13.0      12.0      14.0
     Measure words  5      12.60         13.0      12.0      13.0
 Vocabulary review  8       9.75         10.0       9.0      11.0
        Word order  6       8.83          9.0       8.0       9.0
Saved summary table to: weeks/week-05-visualization-for-paper/outputs/tables/week05_figure_summary.csv


## 6. Figure 1: mean bar chart

Bar chart giúp so sánh group means. Ta thêm N dưới mỗi label để reader không quên sample size.

In [6]:
palette = ["#2563eb", "#1f7a4d", "#b45309", "#b8325f"]
plot_order = [plot_labels[x] for x in order]
bar_png = FIGURE_DIR / "week05_mean_gain_by_activity.png"
bar_svg = FIGURE_DIR / "week05_mean_gain_by_activity.svg"

fig, ax = plt.subplots(figsize=(8.2, 5.2))
sns.barplot(
    data=usable,
    x="plot_label",
    y="gain_score",
    hue="plot_label",
    order=plot_order,
    hue_order=plot_order,
    errorbar=None,
    palette=palette,
    legend=False,
    ax=ax,
)
ax.set_title("Mean gain by activity focus", weight="bold", pad=14)
ax.set_xlabel("Activity focus", labelpad=16)
ax.set_ylabel("Gain score (post - pre)")
ax.set_ylim(0, 16)
ax.tick_params(axis="x", rotation=0)
for patch, (_, row) in zip(ax.patches, summary.iterrows()):
    x = patch.get_x() + patch.get_width() / 2
    y = patch.get_height()
    ax.text(x, y + 0.35, f"{row['mean_gain']:.2f}", ha="center", va="bottom", weight="bold")
tick_labels = [f"{label}\n(n={int(n)})" for label, n in zip(plot_order, summary["n"])]
ax.set_xticks(range(len(plot_order)))
ax.set_xticklabels(tick_labels)
fig.text(0.12, 0.035, "Source: synthetic Week 05 teaching dataset; N = 25 usable records", color="#5b6578", fontsize=10)
fig.tight_layout(rect=(0, 0.11, 1, 1))
fig.savefig(bar_png, dpi=300, bbox_inches="tight")
fig.savefig(bar_svg, bbox_inches="tight")
plt.close(fig)

print("Saved bar PNG:", course_path(bar_png))
print("Saved bar SVG:", course_path(bar_svg))


Saved bar PNG: weeks/week-05-visualization-for-paper/outputs/figures/week05_mean_gain_by_activity.png
Saved bar SVG: weeks/week-05-visualization-for-paper/outputs/figures/week05_mean_gain_by_activity.svg


## 7. Figure 2: individual dot plot

Dot plot cho thấy từng learner record. Với small N, figure này giúp tránh overclaim từ mean bar.

In [7]:
dot_png = FIGURE_DIR / "week05_gain_distribution_by_activity.png"
dot_svg = FIGURE_DIR / "week05_gain_distribution_by_activity.svg"

fig, ax = plt.subplots(figsize=(8.2, 5.2))
sns.stripplot(
    data=usable,
    x="plot_label",
    y="gain_score",
    hue="plot_label",
    order=plot_order,
    hue_order=plot_order,
    palette=palette,
    legend=False,
    jitter=0.12,
    size=8,
    alpha=0.82,
    ax=ax,
)
for idx, row in summary.iterrows():
    ax.hlines(row["mean_gain"], idx - 0.28, idx + 0.28, color="#172033", linewidth=3)
    ax.text(idx, row["mean_gain"] + 0.35, f"mean={row['mean_gain']:.2f}", ha="center", va="bottom", fontsize=9, weight="bold")
ax.set_title("Individual gain scores by activity focus", weight="bold", pad=14)
ax.set_xlabel("Activity focus", labelpad=16)
ax.set_ylabel("Gain score (post - pre)")
ax.set_ylim(0, 16)
ax.tick_params(axis="x", rotation=0)
tick_labels = [f"{label}\n(n={int(n)})" for label, n in zip(plot_order, summary["n"])]
ax.set_xticks(range(len(plot_order)))
ax.set_xticklabels(tick_labels)
fig.text(0.12, 0.035, "Each dot is one usable learner record; horizontal line marks group mean", color="#5b6578", fontsize=10)
fig.tight_layout(rect=(0, 0.11, 1, 1))
fig.savefig(dot_png, dpi=300, bbox_inches="tight")
fig.savefig(dot_svg, bbox_inches="tight")
plt.close(fig)

print("Saved dot PNG:", course_path(dot_png))
print("Saved dot SVG:", course_path(dot_svg))


Saved dot PNG: weeks/week-05-visualization-for-paper/outputs/figures/week05_gain_distribution_by_activity.png
Saved dot SVG: weeks/week-05-visualization-for-paper/outputs/figures/week05_gain_distribution_by_activity.svg


## 8. Caption và interpretation

Caption nói figure show gì. Interpretation nói pattern nghĩa là gì, nhưng phải có limitation.

In [8]:
usable_n = len(usable)
top = summary.sort_values("mean_gain", ascending=False).iloc[0]
low = summary.sort_values("mean_gain", ascending=True).iloc[0]
caption = (
    f"Figure 1. Mean gain score by activity focus in the synthetic Week 05 TCSOL dataset "
    f"(N = {usable_n} usable learner records). Bars show group means after filtering to completed records "
    "with numeric pre/post scores. The figure is descriptive and does not establish causal effects."
)
interpretation = (
    f"The figure suggests that {top['activity_label']} has the highest descriptive mean gain ({top['mean_gain']:.2f}), "
    f"while {low['activity_label']} has the lowest ({low['mean_gain']:.2f}). "
    "The individual-point figure is important because each activity group contains only 5-8 usable learner records, "
    "so the mean bar alone hides the small sample and spread. Therefore, the figure supports a cautious descriptive statement rather than a causal claim."
)
source_note = (
    "Dataset source: Synthetic Week 05 visualization dataset derived from the Week 04 cleaned teaching dataset; "
    "not real student data. Raw file: data/raw/week05_cleaned_tcsol_scores.csv. Access date: 2026-06-03."
)

print("Caption:\n", caption, sep="")
print("\nInterpretation:\n", interpretation, sep="")
print("\nSource note:\n", source_note, sep="")


Caption:
Figure 1. Mean gain score by activity focus in the synthetic Week 05 TCSOL dataset (N = 25 usable learner records). Bars show group means after filtering to completed records with numeric pre/post scores. The figure is descriptive and does not establish causal effects.

Interpretation:
The figure suggests that Result complements has the highest descriptive mean gain (13.00), while Word order has the lowest (8.83). The individual-point figure is important because each activity group contains only 5-8 usable learner records, so the mean bar alone hides the small sample and spread. Therefore, the figure supports a cautious descriptive statement rather than a causal claim.

Source note:
Dataset source: Synthetic Week 05 visualization dataset derived from the Week 04 cleaned teaching dataset; not real student data. Raw file: data/raw/week05_cleaned_tcsol_scores.csv. Access date: 2026-06-03.


## 9. Bài tập nhỏ trong notebook

Đổi một display label trong `activity_labels`, ví dụ `Measure words / lượng từ`, rồi chạy lại từ section 4. Ghi lại: dữ liệu có đổi không, và caption có cần đổi không?